# PsychMonitor — обучение RuBERT-base классификатора триггеров стресса

**Модель:** `DeepPavlov/rubert-base-cased` (110M параметров)  
**Задача:** классификация 6 триггеров стресса (work / family / health / friends / finance / unknown)  

### Порядок запуска
1. **Runtime → Change runtime type → T4 GPU**
2. Запустить все ячейки по порядку (`Runtime → Run all`)
3. После завершения скачать архив `rubert_trigger_classifier.zip` из последней ячейки

In [ ]:
# ── Ячейка 1: проверка GPU ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU не найден — переключи runtime на T4!')

In [ ]:
# ── Ячейка 2: клонируем репо ─────────────────────────────────────────────────
!git clone https://github.com/KyKyPyKy19/Math-model.git
%cd Math-model/psychmonitor_classifier
!ls

In [ ]:
# ── Ячейка 3: устанавливаем зависимости ──────────────────────────────────────
!pip install -q -r requirements.txt
# sentencepiece нужен для токенизатора rubert-base
!pip install -q sentencepiece
print('Зависимости установлены')

In [ ]:
# ── Ячейка 4: проверяем данные ───────────────────────────────────────────────
import pandas as pd

for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/processed/{split}.csv')
    print(f'{split:5s}: {len(df):5d} примеров | классы: {sorted(df["label_id"].unique().tolist())}')

import json
with open('data/processed/class_weights.json') as f:
    print(f'\nclass_weights: {json.load(f)}')

In [ ]:
# ── Ячейка 5: обучение ───────────────────────────────────────────────────────
# Меняем рабочую директорию на src/ чтобы работали относительные импорты
import sys, os
os.chdir('src')
sys.path.insert(0, '.')

# Запускаем обучение
exec(open('train.py').read())
main()

In [ ]:
# ── Ячейка 6: оценка на test-сплите ─────────────────────────────────────────
exec(open('evaluate.py').read())
main()

In [ ]:
# ── Ячейка 7: показываем графики ─────────────────────────────────────────────
from IPython.display import Image, display
import os

reports = '../reports'
for img in ['confusion_matrix.png', 'confidence_histogram.png']:
    path = os.path.join(reports, img)
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(Image(path))

In [ ]:
# ── Ячейка 8: скачиваем обученную модель ─────────────────────────────────────
import shutil
from google.colab import files

model_dir = '../models/rubert_trigger_classifier'
zip_path = '/content/rubert_trigger_classifier'

shutil.make_archive(zip_path, 'zip', model_dir)
print(f'Архив готов: {zip_path}.zip')
files.download(f'{zip_path}.zip')